# ***GÖREV 1***

Tek head'i multi-head attention'a çevir: aynı boyutu n_head tane küçük head'e böl, çıktıları birleştir. Val loss'u geçen haftaki tek head'le yan yana koy. (1:21:59 - 1:24:25)


In [ ]:
!wget  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-10 18:15:00--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-10-10 18:15:01 (51.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0


torch.manual_seed(1337)


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):

        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.sa_head(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):


    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')


    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.022721 M parameters
step 0: train loss 4.1833, val loss 4.1869
step 100: train loss 3.0050, val loss 3.0363
step 200: train loss 2.8227, val loss 2.8392
step 300: train loss 2.7148, val loss 2.7341
step 400: train loss 2.6388, val loss 2.6461
step 500: train loss 2.5939, val loss 2.5951
step 600: train loss 2.5448, val loss 2.5488
step 700: train loss 2.5140, val loss 2.5246
step 800: train loss 2.4861, val loss 2.4868
step 900: train loss 2.4649, val loss 2.4787
step 1000: train loss 2.4572, val loss 2.4733
step 1100: train loss 2.4415, val loss 2.4440
step 1200: train loss 2.4313, val loss 2.4354
step 1300: train loss 2.4174, val loss 2.4277
step 1400: train loss 2.4078, val loss 2.4216
step 1500: train loss 2.4014, val loss 2.4198
step 1600: train loss 2.4052, val loss 2.4133
step 1700: train loss 2.3900, val loss 2.4004
step 1800: train loss 2.3892, val loss 2.4049
step 1900: train loss 2.3843, val loss 2.3943
step 2000: train loss 2.3771, val loss 2.4044
step 2100: train loss 2.

In [ ]:
torch.manual_seed(2026)
single_head_val_loss = estimate_loss()["val"].item()
print(f"Tek-head val loss: {single_head_val_loss:.4f}")

Tek-head val loss: 2.3447


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0


torch.manual_seed(1337)


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):

        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = MultiHeadAttention(n_head, n_embd // n_head)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.sa_head(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):


    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')


    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.022721 M parameters
step 0: train loss 4.1859, val loss 4.1881
step 100: train loss 2.9416, val loss 2.9671
step 200: train loss 2.7456, val loss 2.7597
step 300: train loss 2.6570, val loss 2.6752
step 400: train loss 2.6149, val loss 2.6248
step 500: train loss 2.5869, val loss 2.5880
step 600: train loss 2.5470, val loss 2.5548
step 700: train loss 2.5220, val loss 2.5318
step 800: train loss 2.4941, val loss 2.5034
step 900: train loss 2.4692, val loss 2.4885
step 1000: train loss 2.4549, val loss 2.4701
step 1100: train loss 2.4295, val loss 2.4319
step 1200: train loss 2.4070, val loss 2.4142
step 1300: train loss 2.3849, val loss 2.3975
step 1400: train loss 2.3640, val loss 2.3767
step 1500: train loss 2.3514, val loss 2.3714
step 1600: train loss 2.3427, val loss 2.3574
step 1700: train loss 2.3103, val loss 2.3302
step 1800: train loss 2.3039, val loss 2.3262
step 1900: train loss 2.2890, val loss 2.3077
step 2000: train loss 2.2697, val loss 2.3148
step 2100: train loss 2.

In [ ]:
torch.manual_seed(2026)
multi_head_val_loss = estimate_loss()["val"].item()

print(f"Tek head   : {single_head_val_loss:.4f}")
print(f"Multi-head : {multi_head_val_loss:.4f}")

Tek head   : 2.3447
Multi-head : 2.1515


# ***GÖREV 2***

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0


torch.manual_seed(1337)


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):

        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
      super().__init__()

      self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
      self.position_embedding_table = nn.Embedding(block_size, n_embd)

      self.sa_head = MultiHeadAttention(n_head, n_embd // n_head)
      self.ffwd = FeedFoward(n_embd)
      self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.sa_head(x)
        x = self.ffwd(x)  # YENİ
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):


    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')


    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.055809 M parameters
step 0: train loss 4.1838, val loss 4.1833
step 100: train loss 2.8516, val loss 2.8760
step 200: train loss 2.6717, val loss 2.6850
step 300: train loss 2.5874, val loss 2.5926
step 400: train loss 2.5265, val loss 2.5235
step 500: train loss 2.4669, val loss 2.4777
step 600: train loss 2.4227, val loss 2.4283
step 700: train loss 2.3767, val loss 2.3736
step 800: train loss 2.3364, val loss 2.3441
step 900: train loss 2.3025, val loss 2.3118
step 1000: train loss 2.2694, val loss 2.2810
step 1100: train loss 2.2381, val loss 2.2520
step 1200: train loss 2.2298, val loss 2.2382
step 1300: train loss 2.1990, val loss 2.2233
step 1400: train loss 2.1770, val loss 2.1965
step 1500: train loss 2.1624, val loss 2.1868
step 1600: train loss 2.1291, val loss 2.1778
step 1700: train loss 2.1128, val loss 2.1629
step 1800: train loss 2.0993, val loss 2.1411
step 1900: train loss 2.0823, val loss 2.1330
step 2000: train loss 2.0689, val loss 2.1290
step 2100: train loss 2.

In [ ]:
torch.manual_seed(2026)
feedforward_val_loss = estimate_loss()["val"].item()

print(f"Tek head               : {single_head_val_loss:.4f}")
print(f"Multi-head             : {multi_head_val_loss:.4f}")
print(f"Multi-head + Feedforward: {feedforward_val_loss:.4f}")

Tek head               : 2.3447
Multi-head             : 2.1515
Multi-head + Feedforward: 1.9750


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0


torch.manual_seed(1337)


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):

        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedFoward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = x + self.sa_head(x)  # Residual
        x = x + self.ffwd(x)     # Residual
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):


    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')


    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.055809 M parameters
step 0: train loss 4.5491, val loss 4.5445
step 100: train loss 2.7224, val loss 2.7435
step 200: train loss 2.5751, val loss 2.5870
step 300: train loss 2.4969, val loss 2.5061
step 400: train loss 2.4385, val loss 2.4393
step 500: train loss 2.3760, val loss 2.3927
step 600: train loss 2.3387, val loss 2.3541
step 700: train loss 2.2946, val loss 2.3045
step 800: train loss 2.2514, val loss 2.2782
step 900: train loss 2.2270, val loss 2.2476
step 1000: train loss 2.1861, val loss 2.2116
step 1100: train loss 2.1661, val loss 2.1904
step 1200: train loss 2.1511, val loss 2.1693
step 1300: train loss 2.1212, val loss 2.1629
step 1400: train loss 2.1090, val loss 2.1416
step 1500: train loss 2.0957, val loss 2.1336
step 1600: train loss 2.0692, val loss 2.1312
step 1700: train loss 2.0560, val loss 2.1144
step 1800: train loss 2.0502, val loss 2.1005
step 1900: train loss 2.0346, val loss 2.0956
step 2000: train loss 2.0254, val loss 2.0952
step 2100: train loss 2.

In [ ]:
torch.manual_seed(2026)
residual_val_loss = estimate_loss()["val"].item()

print(f"Tek head                : {single_head_val_loss:.4f}")
print(f"Multi-head              : {multi_head_val_loss:.4f}")
print(f"Multi-head + Feedforward : {feedforward_val_loss:.4f}")
print(f"+ Residual              : {residual_val_loss:.4f}")

Tek head                : 2.3447
Multi-head              : 2.1515
Multi-head + Feedforward : 1.9750
+ Residual              : 1.9792


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0


torch.manual_seed(1337)


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.sa_head = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedFoward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.sa_head = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedFoward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)   # Attention öncesi
        self.ln2 = nn.LayerNorm(n_embd)   # Feedforward öncesi
        self.ln_f = nn.LayerNorm(n_embd)  # Çıkış katmanı öncesi

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = tok_emb + pos_emb
        x = x + self.sa_head(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        x = self.ln_f(x)

        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):


    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')


    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.056193 M parameters
step 0: train loss 4.3298, val loss 4.3260
step 100: train loss 2.8076, val loss 2.8290
step 200: train loss 2.5945, val loss 2.6033
step 300: train loss 2.5078, val loss 2.5158
step 400: train loss 2.4435, val loss 2.4459
step 500: train loss 2.3820, val loss 2.3960
step 600: train loss 2.3446, val loss 2.3610
step 700: train loss 2.2958, val loss 2.3061
step 800: train loss 2.2625, val loss 2.2845
step 900: train loss 2.2312, val loss 2.2526
step 1000: train loss 2.1969, val loss 2.2166
step 1100: train loss 2.1735, val loss 2.1991
step 1200: train loss 2.1548, val loss 2.1739
step 1300: train loss 2.1275, val loss 2.1626
step 1400: train loss 2.1107, val loss 2.1370
step 1500: train loss 2.0964, val loss 2.1286
step 1600: train loss 2.0697, val loss 2.1278
step 1700: train loss 2.0538, val loss 2.1097
step 1800: train loss 2.0460, val loss 2.0947
step 1900: train loss 2.0315, val loss 2.0894
step 2000: train loss 2.0176, val loss 2.0860
step 2100: train loss 2.

In [ ]:
torch.manual_seed(2026)
layernorm_val_loss = estimate_loss()["val"].item()

print(f"Tek head                : {single_head_val_loss:.4f}")
print(f"Multi-head              : {multi_head_val_loss:.4f}")
print(f"Multi-head + Feedforward : {feedforward_val_loss:.4f}")
print(f"+ Residual              : {residual_val_loss:.4f}")
print(f"+ LayerNorm             : {layernorm_val_loss:.4f}")

Tek head                : 2.3447
Multi-head              : 2.1515
Multi-head + Feedforward : 1.9750
+ Residual              : 1.9792
+ LayerNorm             : 1.9686


In [ ]:
# B: örnek sayısı, T: token sayısı, C: embedding boyutu
x = torch.randn(2, 3, 4)  # (B, T, C)

# LayerNorm: her token'ın kendi 4 özelliği üzerinden
ln_mean = x.mean(dim=-1, keepdim=True)

# Önceki görevde düzelttiğimiz BatchNorm:
# her özellik için bütün örnekler ve zaman konumları üzerinden
bn_mean = x.mean(dim=(0, 1), keepdim=True)

print("Girdi:", x.shape)
print("LayerNorm ortalaması:", ln_mean.shape)
print("BatchNorm ortalaması:", bn_mean.shape)

Girdi: torch.Size([2, 3, 4])
LayerNorm ortalaması: torch.Size([2, 3, 1])
BatchNorm ortalaması: torch.Size([1, 1, 4])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0


torch.manual_seed(1337)


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
      logits = self.token_embedding_table(idx)

      loss = None
      if targets is not None:
        B, T, C = logits.shape
        loss = F.cross_entropy(
            logits.reshape(B * T, C),
            targets.reshape(B * T)
        )

      return logits, loss
    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):


    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')


    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.004225 M parameters
step 0: train loss 4.7260, val loss 4.7259
step 100: train loss 4.5986, val loss 4.5953
step 200: train loss 4.4763, val loss 4.4755
step 300: train loss 4.3572, val loss 4.3574
step 400: train loss 4.2446, val loss 4.2459
step 500: train loss 4.1332, val loss 4.1303
step 600: train loss 4.0279, val loss 4.0317
step 700: train loss 3.9268, val loss 3.9321
step 800: train loss 3.8344, val loss 3.8387
step 900: train loss 3.7425, val loss 3.7476
step 1000: train loss 3.6578, val loss 3.6587
step 1100: train loss 3.5823, val loss 3.5821
step 1200: train loss 3.5049, val loss 3.5055
step 1300: train loss 3.4314, val loss 3.4358
step 1400: train loss 3.3648, val loss 3.3694
step 1500: train loss 3.2985, val loss 3.3034
step 1600: train loss 3.2409, val loss 3.2487
step 1700: train loss 3.1900, val loss 3.1855
step 1800: train loss 3.1357, val loss 3.1408
step 1900: train loss 3.0850, val loss 3.0887
step 2000: train loss 3.0422, val loss 3.0514
step 2100: train loss 3.

In [ ]:
torch.manual_seed(2026)
bigram_val_loss = estimate_loss()["val"].item()

print(f"Bigram val loss: {bigram_val_loss:.4f}")

Bigram val loss: 2.5390


In [ ]:
import pandas as pd

sonuclar = pd.DataFrame({
    "Model": [
        "Bigram",
        "Tek head",
        "Multi-head",
        "Multi-head + Feedforward",
        "+ Residual",
        "+ LayerNorm"
    ],
    "Val loss": [
        bigram_val_loss,
        single_head_val_loss,
        multi_head_val_loss,
        feedforward_val_loss,
        residual_val_loss,
        layernorm_val_loss
    ]
})

display(sonuclar.round(4))

sonuclar.to_csv("gorev2_val_loss_karsilastirma.csv", index=False)

,Model,Val loss
0,Bigram,2.5390
1,Tek head,2.3447
2,Multi-head,2.1515
3,Multi-head + Feedforward,1.9750
4,+ Residual,1.9792
5,+ LayerNorm,1.9686


# ***GÖREV 3***

In [ ]:
print("Cihaz:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    bellek = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU belleği: {bellek / 1024**3:.1f} GB")

Cihaz: cuda
GPU: Tesla T4
GPU belleği: 14.6 GB


In [ ]:
batch_size = 32
block_size = 128
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 100

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2

In [ ]:
import json
import time
import torch
import torch.nn as nn
from torch.nn import functional as F

# ── 1. Ayarlar ────────────────────────────────────────────

batch_size = 32
block_size = 128
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 100

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(1337)
assert n_embd % n_head == 0

# ── 2. Veriyi hazırla ─────────────────────────────────────

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(indices):
    return "".join(itos[i] for i in indices)

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))

    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])

    return x.to(device), y.to(device)

# ── 3. Tek attention head ─────────────────────────────────

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        return wei @ v

# ── 4. Multi-head attention ───────────────────────────────

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        # Birleştirilmiş head çıktılarını karıştırır.
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.proj(out)
        return self.dropout(out)

# ── 5. Feedforward ───────────────────────────────────────

class FeedFoward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# ── 6. Transformer bloğu ─────────────────────────────────

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# ── 7. Tam model ─────────────────────────────────────────

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )
        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        # Her biri ayrı parametrelere sahip dört blok.
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head) for _ in range(n_layer)
        ])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(B * T, vocab_size),
                targets.reshape(B * T)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# ── 8. Model ve değerlendirme ─────────────────────────────

model = GPTLanguageModel().to(device)

parameter_count = sum(p.numel() for p in model.parameters())

print("Cihaz:", device)
print(f"Parametre sayısı: {parameter_count:,}")
print(f"Blok sayısı: {len(model.blocks)}")

@torch.no_grad()
def estimate_loss():
    out = {}
    was_training = model.training
    model.eval()  # Değerlendirmede dropout kapanır.

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean().item()

    model.train(was_training)
    return out

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

# ── 9. Eğitim ve süre ölçümü ──────────────────────────────

if device == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()
model.train()

for step in range(max_iters):
    if step % eval_interval == 0:
        losses = estimate_loss()
        print(
            f"step {step}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

if device == "cuda":
    torch.cuda.synchronize()

# Eğitim döngüsü içindeki ara değerlendirmeler de dahildir.
training_seconds = time.perf_counter() - start_time

# Son güncellemeden sonra nihai ölçüm.
torch.manual_seed(2026)
final_losses = estimate_loss()
gpt_val_loss = final_losses["val"]

print(f"\nSon train loss: {final_losses['train']:.4f}")
print(f"Son val loss:   {gpt_val_loss:.4f}")
print(f"Eğitim süresi:  {training_seconds / 60:.2f} dakika")
print("(Süreye ara değerlendirmeler dahildir.)")

# ── 10. Metin üretimi ─────────────────────────────────────

model.eval()  # Metin üretirken de dropout kapanır.
torch.manual_seed(2026)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=2000)
generated_text = decode(generated[0].tolist())

print("\nÜRETİLEN METİN:\n")
print(generated_text)

# ── 11. Sonuçları dosyaya kaydet ──────────────────────────

results = {
    "model": "4 bloklu GPT",
    "device": device,
    "gpu": (
        torch.cuda.get_device_name(0)
        if device == "cuda" else None
    ),
    "batch_size": batch_size,
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,
    "learning_rate": learning_rate,
    "max_iters": max_iters,
    "vocab_size": vocab_size,
    "parameter_count": parameter_count,
    "train_loss": final_losses["train"],
    "val_loss": gpt_val_loss,
    "training_seconds_including_evaluations": training_seconds,
}

with open("gorev3_sonuclar.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open("gorev3_uretilen_metin.txt", "w", encoding="utf-8") as f:
    f.write(generated_text)

print("\nSonuçlar ve üretilen metin kaydedildi.")

Cihaz: cuda
Parametre sayısı: 824,897
Blok sayısı: 4
step 0: train loss 4.3491, val loss 4.3464
step 500: train loss 2.4080, val loss 2.4211
step 1000: train loss 2.1588, val loss 2.1896
step 1500: train loss 1.9948, val loss 2.0512
step 2000: train loss 1.8777, val loss 1.9714
step 2500: train loss 1.7927, val loss 1.9081
step 3000: train loss 1.7315, val loss 1.8749
step 3500: train loss 1.6885, val loss 1.8386
step 4000: train loss 1.6450, val loss 1.8010
step 4500: train loss 1.6121, val loss 1.7790

Son train loss: 1.5875
Son val loss:   1.7591
Eğitim süresi:  2.38 dakika
(Süreye ara değerlendirmeler dahildir.)

ÜRETİLEN METİN:


KING RICHEMILLORD:
The shall, thou overy wear dostedgal'st gent
And Macious proyongfep; jesice is the displessed.

LEONTES:
When, when some are seemion.

GLOUCESTER:
Entune propency; by chomment, I repond,
But addme; in I foul your rome. God. Easter' had all Lamidy;
'Which that as was I propincant unstritorn,
And she than would countreamen the hands this 

# ***GÖREV 4***

In [ ]:
NN%pip -q install requests beautifulsoup4

import requests
import time
import re
import unicodedata
import hashlib
import json
from bs4 import BeautifulSoup
from urllib.parse import unquote, quote

API = "https://tr.wikisource.org/w/api.php"

session = requests.Session()
session.headers.update({
    "User-Agent": "YZ50StudentCorpus/1.0 (educational text collection)"
})

def sayfayi_al(baslik):
    response = session.get(
        API,
        params={
            "action": "parse",
            "page": baslik,
            "prop": "text",
            "format": "json",
            "formatversion": 2,
            "redirects": 1,
        },
        timeout=60,
    )
    response.raise_for_status()
    result = response.json()

    if "error" in result:
        raise RuntimeError(result["error"])

    return BeautifulSoup(result["parse"]["text"], "html.parser")


yazar = sayfayi_al("Kişi:Ömer Seyfettin")

basliklar = []
hikaye_bolumunde = False

for element in yazar.find_all(["h2", "a"]):
    if element.name == "h2":
        heading = element.get_text(" ", strip=True)

        if "Hikâyeleri" in heading:
            hikaye_bolumunde = True
        elif hikaye_bolumunde:
            break

    elif hikaye_bolumunde:
        href = element.get("href", "")

        if not href.startswith("/wiki/"):
            continue
        if "new" in element.get("class", []):
            continue

        baslik = unquote(href.split("/wiki/", 1)[1])
        baslik = baslik.split("#")[0].replace("_", " ")


        if ":" not in baslik and baslik not in basliklar:
            basliklar.append(baslik)

if not basliklar:
    raise RuntimeError("Hikâye bağlantıları bulunamadı.")

print("Bulunan hikâye bağlantısı:", len(basliklar))


def metni_temizle(soup):
    for element in soup.select(
        "table, .navbox, .metadata, .ambox, "
        ".ws-license, .licenseContainer, "
        ".mw-editsection, .reference, .references, #toc"
    ):
        element.decompose()

    for br in soup.find_all("br"):
        br.replace_with("\n")

    paragraflar = []

    for p in soup.find_all("p"):
        text = p.get_text()
        text = unicodedata.normalize("NFC", text)
        text = text.replace("\xa0", " ")
        text = re.sub(r"[ \t]+", " ", text).strip()

        if text:
            paragraflar.append(text)

    return "\n\n".join(paragraflar)

hedef_bayt = 1_000_000
toplam_bayt = 0

metinler = []
kaynaklar = []
gorulen_metinler = set()

for baslik in basliklar:
    try:
        soup = sayfayi_al(baslik)
        metin = metni_temizle(soup)


        if len(metin) < 500:
            continue


        ozet = hashlib.sha256(metin.encode("utf-8")).hexdigest()
        if ozet in gorulen_metinler:
            continue

        gorulen_metinler.add(ozet)

        parca = baslik + "\n\n" + metin
        metinler.append(parca)
        toplam_bayt += len((parca + "\n\n").encode("utf-8"))

        kaynaklar.append({
            "eser": baslik,
            "yazar": "Ömer Seyfettin",
            "url": (
                "https://tr.wikisource.org/wiki/"
                + quote(baslik.replace(" ", "_"), safe="")
            ),
        })

        print(f"{len(metinler):>3}. {baslik} — {toplam_bayt / 1e6:.2f} MB")

        if toplam_bayt >= hedef_bayt:
            break

    except (requests.RequestException, RuntimeError, ValueError) as error:
        print(f"Alınamadı: {baslik} — {error}")

    finally:
        time.sleep(0.5)



if not metinler:
    raise RuntimeError("Metin indirilemedi; eğitim aşamasına geçme.")

turkce_metin = "\n\n".join(metinler)

with open("input_tr.txt", "w", encoding="utf-8") as f:
    f.write(turkce_metin)

with open("turkce_kaynaklar.json", "w", encoding="utf-8") as f:
    json.dump(kaynaklar, f, ensure_ascii=False, indent=2)

print("\n--- VERİ ÖZETİ ---")
print("Hikâye sayısı:", len(metinler))
print("Dosya boyutu:", round(len(turkce_metin.encode("utf-8")) / 1e6, 3), "MB")
print("Karakter sayısı:", len(turkce_metin))
print("Farklı karakter sayısı:", len(set(turkce_metin)))

print("\n--- İLK 1000 KARAKTER ---")
print(turkce_metin[:1000])

if toplam_bayt < hedef_bayt:
    print("\nHedef boyuta ulaşılamadı; ek eser seçmemiz gerekiyor.")

Bulunan hikâye bağlantısı: 64
  1. Antiseptik — 0.00 MB
  2. Aşk Dalgası — 0.02 MB
  3. Aşk ve Ayak Parmakları — 0.03 MB
  4. Balkon (hikâye) — 0.04 MB
  5. Başını Vermeyen Şehit — 0.05 MB
  6. Binecek Şey — 0.07 MB
  7. Bir Hatıra — 0.07 MB
  8. Bir Kayışın Tesiri — 0.08 MB
  9. Bir Refikin Defter-i İhtisasatından — 0.08 MB
Alınamadı: Bomba — 429 Client Error: Too Many Requests for url: https://tr.wikisource.org/w/api.php?action=parse&page=Bomba&prop=text&format=json&formatversion=2&redirects=1
Alınamadı: Buse-i Mader — 429 Client Error: Too Many Requests for url: https://tr.wikisource.org/w/api.php?action=parse&page=Buse-i+Mader&prop=text&format=json&formatversion=2&redirects=1
Alınamadı: Büyücü — 429 Client Error: Too Many Requests for url: https://tr.wikisource.org/w/api.php?action=parse&page=B%C3%BCy%C3%BCc%C3%BC&prop=text&format=json&formatversion=2&redirects=1
Alınamadı: Cimnastiğe Dair - 2 — 429 Client Error: Too Many Requests for url: https://tr.wikisource.org/w/api.php?action

In [ ]:
from pathlib import Path

cache = Path("turkce_metin_cache")
cache.mkdir(exist_ok=True)

def metni_temizle_duzeltilmis(soup):
    for element in soup.select(
        "table, .navbox, .metadata, .ambox, "
        ".ws-license, .licenseContainer, "
        ".mw-editsection, .reference, .references, #toc"
    ):
        element.decompose()

    for br in soup.find_all("br"):
        br.replace_with("\n")

    parcalar = []

    etiketler = ["p", "dd", "dt", "li"]

    for element in soup.find_all(etiketler):

        if element.find_parent(etiketler) is not None:
            continue

        metin = element.get_text()
        metin = unicodedata.normalize("NFC", metin)
        metin = metin.replace("\xa0", " ")
        metin = re.sub(r"[ \t]+", " ", metin).strip()

        if metin:
            parcalar.append(metin)

    return "\n\n".join(parcalar)


metinler = []
kaynaklar = []
gorulen_metinler = set()
toplam_bayt = 0

for baslik in basliklar:
    dosya_adi = hashlib.sha256(
        baslik.encode("utf-8")
    ).hexdigest() + ".txt"

    cache_dosyasi = cache / dosya_adi

    if cache_dosyasi.exists():
        metin = cache_dosyasi.read_text(encoding="utf-8")

    else:

        time.sleep(10)

        try:
            soup = sayfayi_al(baslik)
            metin = metni_temizle_duzeltilmis(soup)
            cache_dosyasi.write_text(metin, encoding="utf-8")

        except requests.HTTPError as error:
            if error.response.status_code == 429:
                print("\nSite yeniden hız sınırı uyguladı; durduruldu.")
                print("İnen metinler önbellekte duruyor.")
                print(
                    "Sunucunun Retry-After bilgisi:",
                    error.response.headers.get("Retry-After", "Belirtilmemiş")
                )
                break

            print("Alınamadı:", baslik)
            continue

        except (requests.RequestException, RuntimeError, ValueError) as error:
            print("Alınamadı:", baslik, str(error)[:120])
            continue

    if len(metin) < 500:
        continue

    ozet = hashlib.sha256(metin.encode("utf-8")).hexdigest()
    if ozet in gorulen_metinler:
        continue

    gorulen_metinler.add(ozet)
    parca = baslik + "\n\n" + metin
    metinler.append(parca)

    toplam_bayt += len((parca + "\n\n").encode("utf-8"))

    kaynaklar.append({
        "eser": baslik,
        "yazar": "Ömer Seyfettin",
        "url": "https://tr.wikisource.org/wiki/"
               + quote(baslik.replace(" ", "_"), safe=""),
    })

    print(f"{len(metinler)}. {baslik}: {toplam_bayt / 1e6:.3f} MB")

    if toplam_bayt >= 1_000_000:
        break


if metinler:
    turkce_metin = "\n\n".join(metinler)

    Path("input_tr.txt").write_text(turkce_metin, encoding="utf-8")

    with open("turkce_kaynaklar.json", "w", encoding="utf-8") as f:
        json.dump(kaynaklar, f, ensure_ascii=False, indent=2)

    print("\n--- VERİ ÖZETİ ---")
    print("Hikâye sayısı:", len(metinler))
    print("Boyut:", round(len(turkce_metin.encode("utf-8")) / 1e6, 3), "MB")
    print("Karakter sayısı:", len(turkce_metin))
    print("Farklı karakter sayısı:", len(set(turkce_metin)))

    print("\n--- İLK 1000 KARAKTER ---")
    print(turkce_metin[:1000])

    if toplam_bayt < 1_000_000:
        print("\nHenüz hedef boyutta değil; eğitime geçme.")
else:
    print("Yeni metin alınamadı. Mevcut dosyan değiştirilmedi.")

1. Antiseptik: 0.005 MB
2. Aşk Dalgası: 0.027 MB
3. Aşk ve Ayak Parmakları: 0.035 MB
4. Balkon (hikâye): 0.050 MB
5. Başını Vermeyen Şehit: 0.070 MB
6. Binecek Şey: 0.081 MB
7. Bir Hatıra: 0.091 MB
8. Bir Kayışın Tesiri: 0.096 MB
9. Bir Refikin Defter-i İhtisasatından: 0.098 MB
10. Bomba: 0.134 MB
11. Buse-i Mader: 0.135 MB
12. Büyücü: 0.150 MB
13. Cimnastiğe Dair - 2: 0.165 MB
14. Çakmak: 0.173 MB
15. Çirkinliğin Esrarı: 0.186 MB
16. Diyet: 0.205 MB
17. Elma: 0.209 MB
18. Falaka: 0.222 MB
19. Ferman: 0.246 MB
20. Fon Sadriştayn’ın Karısı: 0.271 MB
21. Fon Sadriştayn’ın Oğlu: 0.297 MB
22. Forsa: 0.306 MB
23. Havyar: 0.313 MB
24. Hediye...: 0.317 MB
25. Hürriyet Bayrakları: 0.335 MB
26. İffet: 0.336 MB
27. İlk Cinayet: 0.342 MB
28. İlk Namaz: 0.351 MB
29. İki Mebus: 0.367 MB
30. İlkbahar: 0.370 MB
31. Kaşağı: 0.377 MB
32. Keramet: 0.381 MB
33. Kesik Bıyık: 0.386 MB
34. Kır Sineği: 0.388 MB
35. Kıskançlık: 0.401 MB
36. Korkunç Bir Ceza: 0.408 MB
37. Kurbağa Duası: 0.420 MB
38. Kurumuş Ağ

In [ ]:
from pathlib import Path
import json
import re

# Ömer Seyfettin DERLEMESİ
omer_dosyasi = Path("input_tr_omer.txt")

if not omer_dosyasi.exists():
    omer_dosyasi.write_text(
        Path("input_tr.txt").read_text(encoding="utf-8"),
        encoding="utf-8"
    )

omer_metni = omer_dosyasi.read_text(encoding="utf-8")

roman_dosyasi = Path("kurk_mantolu_madonna.txt")

if roman_dosyasi.exists():
    roman_metni = roman_dosyasi.read_text(encoding="utf-8")
else:
    soup = sayfayi_al("Kürk Mantolu Madonna")

    for element in soup.select(
        ".pagenum, .pagenumber, .mw-cite-backlink, "
        ".wst-running-header, .wst-running-footer"
    ):
        element.decompose()

    roman_metni = metni_temizle_duzeltilmis(soup)


    roman_metni = roman_metni.replace("\u00ad", "")
    roman_metni = roman_metni.replace("\u200b", "")
    roman_metni = re.sub(r"[^\S\n]+", " ", roman_metni)


    baslangic = re.search(
        r"[ŞS]imdiye kadar tesadüf",
        roman_metni
    )
    if baslangic:
        roman_metni = roman_metni[baslangic.start():]

    if len(roman_metni) < 10_000:
        raise ValueError("Roman beklenenden kısa; metni kontrol edelim.")

    roman_dosyasi.write_text(roman_metni, encoding="utf-8")


turkce_metin = (
    omer_metni
    + "\n\nKürk Mantolu Madonna\n\n"
    + roman_metni
)

Path("input_tr.txt").write_text(turkce_metin, encoding="utf-8")

with open("turkce_kaynaklar.json", encoding="utf-8") as f:
    kaynaklar = json.load(f)

if not any(k["eser"] == "Kürk Mantolu Madonna" for k in kaynaklar):
    kaynaklar.append({
        "eser": "Kürk Mantolu Madonna",
        "yazar": "Sabahattin Ali",
        "url": "https://tr.wikisource.org/wiki/Kürk_Mantolu_Madonna"
    })

with open("turkce_kaynaklar.json", "w", encoding="utf-8") as f:
    json.dump(kaynaklar, f, ensure_ascii=False, indent=2)


shakespeare = Path("input.txt").read_text(encoding="utf-8")

print("--- VERİ ÖZETİ ---")
print(f"Türkçe boyut: {len(turkce_metin.encode('utf-8')) / 1e6:.3f} MB")
print("Türkçe toplam karakter:", len(turkce_metin))
print("Türkçe farklı karakter:", len(set(turkce_metin)))
print("Shakespeare farklı karakter:", len(set(shakespeare)))

print("\n--- EKLENEN ROMANIN BAŞLANGICI ---")
print(roman_metni[:1000])

--- VERİ ÖZETİ ---
Türkçe boyut: 0.969 MB
Türkçe toplam karakter: 877308
Türkçe farklı karakter: 107
Shakespeare farklı karakter: 65

--- EKLENEN ROMANIN BAŞLANGICI ---
Şimdiye kadar tesadüf ettiğim Meryemana tasvirlerinde, lüzumundan biraz fazla tebarüz ettirilen, hattâ mânasızlığa kadar götürülen bir masumluk ifadesi vardı; kucaklarındaki bebeğe bakarken: «Gördünüz mü? Allah Baba bana neler ihsan etti!» demek istiyen küçük çocuklara; veya ismini söyliyemiyecekleri bir adamdan peydahladıkları evlâtlarına gözlerini dikip şaşkın şaşkın gülümsiyen hizmetçi kızlara benzerlerdi. Halbuki Sarto’nun bu tablosundaki Meryem, düşünmeği öğrenmiş, hayat hakkındaki hükümlerini vermiş ve dünyayı istihfaf etmiye başlamış bir kadındı. İki tarafında ibadet eder gibi duran azizlere değil, kucağındaki Mesihe değil, hattâ gökyüzüne de değil, toprağa bakıyor ve muhakkak ki bir şeyler görüyordu. 

 Resmi masanın üzerine bıraktım. Gözlerimi kapıyarak sergideki tabloyu düşündüm. Orada tasvir edilen insanın ha

In [ ]:

soup = sayfayi_al("Kürk Mantolu Madonna")
govde = soup.select_one(".mw-parser-output")
if govde is None:
    govde = soup

for element in govde.select(
    "script, style, table, .navbox, .metadata, "
    ".ws-license, .licenseContainer, .mw-editsection, "
    ".pagenum, .pagenumber, .reference, .references, "
    ".wst-running-header, .wst-running-footer, #toc"
):
    element.decompose()


for br in govde.find_all("br"):
    br.replace_with("\n")

for element in govde.find_all(["p", "div", "dd", "li"]):
    element.append("\n")


roman_metni = govde.get_text(separator="")
roman_metni = unicodedata.normalize("NFC", roman_metni)
roman_metni = roman_metni.replace("\u00ad", "")
roman_metni = roman_metni.replace("\u200b", "")
roman_metni = re.sub(r"[^\S\n]+", " ", roman_metni)
roman_metni = re.sub(r"\n\s*\n+", "\n\n", roman_metni)

baslangic = re.search(
    r"[ŞS]imdiye\s+kadar\s+tesadüf\s+ettiğim\s+insanlardan",
    roman_metni
)

if baslangic is None:
    raise ValueError("Romanın gerçek başlangıcı bulunamadı; dosyalar değiştirilmedi.")

roman_metni = roman_metni[baslangic.start():].strip()

omer_metni = Path("input_tr_omer.txt").read_text(encoding="utf-8")
turkce_metin = omer_metni + "\n\nKürk Mantolu Madonna\n\n" + roman_metni

Path("kurk_mantolu_madonna.txt").write_text(
    roman_metni, encoding="utf-8"
)
Path("input_tr.txt").write_text(turkce_metin, encoding="utf-8")

print("--- VERİ ÖZETİ ---")
print(f"Türkçe boyut: {len(turkce_metin.encode('utf-8')) / 1e6:.3f} MB")
print("Toplam karakter:", len(turkce_metin))
print("Farklı karakter:", len(set(turkce_metin)))

print("\n--- ROMANIN BAŞLANGICI ---")
print(roman_metni[:500])

print("\n--- ROMANIN SONU ---")
print(roman_metni[-500:])

--- VERİ ÖZETİ ---
Türkçe boyut: 1.080 MB
Toplam karakter: 978548
Farklı karakter: 108

--- ROMANIN BAŞLANGICI ---
Simdiye kadar tesadüf ettiğim insanlardan bir tanesi benim üzerimde belki en büyük tesiri yapmıştır. Aradan aylar geçtiği halde bir türlü bu tesirden kurtulamadım. Ne zaman kendimle başbaşa kalsam, Raif Efendinin saf yüzü, biraz dünyadan uzak, buna rağmen bir insana tesadüf ettikleri zaman şaşkınca tebessüm etmek istiyen bakışları gözlerimin önünde canlanıyor. Halbuki o hiç de fevkalâde bir adam değildi. Hattâ pek alelâde, hiçbir hususiyeti olmayan, her gün etrafımızda yüzlercesini görüp de bakm

--- ROMANIN SONU ---
kaybetmiş gibi değil, asıl şimdi bulmuş gibi bir his vardı. 

Dün akşam bana: «Seninle şöyle bir oturup konuşamadık!» demişti. Ben artık böyle düşünmüyordum. Dün akşam onunla uzun uzun konuşmuştum.

O bu dünyadan ayrılırken, benim hayatıma, başka hiçbir insana nasip olmıyacak kadar canlı bir şekilde giriyordu. Bundan sonra onu daima yanımda bulacaktım. 

Şirke

In [ ]:
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 32
block_size = 128
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 100

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(1337)
assert n_embd % n_head == 0


with open("input_tr.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(indices):
    return "".join(itos[i] for i in indices)

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

shakespeare_text = Path("input.txt").read_text(encoding="utf-8")
shakespeare_vocab_size = len(set(shakespeare_text))

print(f"Türkçe dosya boyutu: {len(text.encode('utf-8')) / 1e6:.3f} MB")
print("Türkçe toplam karakter:", len(text))
print("Türkçe farklı karakter:", vocab_size)
print("Shakespeare farklı karakter:", shakespeare_vocab_size)

def get_batch(split):
    source = train_data if split == "train" else val_data

    ix = torch.randint(
        len(source) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + block_size] for i in ix
    ])

    y = torch.stack([
        source[i + 1:i + block_size + 1] for i in ix
    ])

    return x.to(device), y.to(device)

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        return wei @ v

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        out = self.proj(out)
        return self.dropout(out)

# 5. Feedforward
class FeedFoward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head)
            for _ in range(n_layer)
        ])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(B * T, vocab_size),
                targets.reshape(B * T)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat((idx, idx_next), dim=1)

        return idx


model = GPTLanguageModel().to(device)

parameter_count = sum(
    p.numel() for p in model.parameters()
)

print("\nCihaz:", device)
print(f"Parametre sayısı: {parameter_count:,}")
print("Blok sayısı:", len(model.blocks))

@torch.no_grad()
def estimate_loss():
    out = {}
    was_training = model.training

    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean().item()

    model.train(was_training)

    return out

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

if device == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()
model.train()

for step in range(max_iters):
    if step % eval_interval == 0:
        losses = estimate_loss()

        print(
            f"step {step}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

if device == "cuda":
    torch.cuda.synchronize()

training_seconds = time.perf_counter() - start_time

torch.manual_seed(2026)

final_losses = estimate_loss()
turkish_val_loss = final_losses["val"]

print(f"\nSon train loss: {final_losses['train']:.4f}")
print(f"Son val loss:   {turkish_val_loss:.4f}")
print(f"Eğitim süresi:  {training_seconds / 60:.2f} dakika")
print("(Süreye ara değerlendirmeler dahildir.)")


model.eval()
torch.manual_seed(2026)

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated = model.generate(
    context,
    max_new_tokens=2000
)

generated_text = decode(generated[0].tolist())

print("\nÜRETİLEN TÜRKÇE METİN:\n")
print(generated_text)


results = {
    "model": "4 bloklu GPT - Türkçe",
    "dataset": "input_tr.txt",
    "device": device,
    "gpu": (
        torch.cuda.get_device_name(0)
        if device == "cuda" else None
    ),
    "batch_size": batch_size,
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,
    "learning_rate": learning_rate,
    "max_iters": max_iters,
    "total_characters": len(text),
    "dataset_bytes": len(text.encode("utf-8")),
    "vocab_size": vocab_size,
    "shakespeare_vocab_size": shakespeare_vocab_size,
    "parameter_count": parameter_count,
    "train_loss": final_losses["train"],
    "val_loss": turkish_val_loss,
    "training_seconds_including_evaluations": training_seconds,
    "validation_split": "Metnin son %10'u; Kürk Mantolu Madonna bölümü",
}

with open("gorev4_sonuclar.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open("gorev4_uretilen_metin.txt", "w", encoding="utf-8") as f:
    f.write(generated_text)

print("\nKaydedildi:")
print("gorev4_sonuclar.json")
print("gorev4_uretilen_metin.txt")

Türkçe dosya boyutu: 1.080 MB
Türkçe toplam karakter: 978548
Türkçe farklı karakter: 108
Shakespeare farklı karakter: 65

Cihaz: cuda
Parametre sayısı: 835,948
Blok sayısı: 4
step 0: train loss 4.8534, val loss 4.8530
step 500: train loss 2.4527, val loss 2.4329
step 1000: train loss 2.1592, val loss 2.1211
step 1500: train loss 2.0393, val loss 1.9986
step 2000: train loss 1.9482, val loss 1.9099
step 2500: train loss 1.8729, val loss 1.8299
step 3000: train loss 1.8199, val loss 1.7634
step 3500: train loss 1.7622, val loss 1.7151
step 4000: train loss 1.7205, val loss 1.6745
step 4500: train loss 1.6820, val loss 1.6371

Son train loss: 1.6578
Son val loss:   1.6114
Eğitim süresi:  2.49 dakika
(Süreye ara değerlendirmeler dahildir.)

ÜRETİLEN TÜRKÇE METİN:


— Pehir yaştığı boş sıkış sürün mavahiz mi?

— Sevkiz karıyor, Ebi haldebile dakalar bir Âda ve takaranlı, benim çürüyor, onlarımı açılmışça yüzerin ağlûdeceğiz...

Ah! diye dal ağacak!

Diye parla dayapmak olduğunu indine sana 